# Prompt Engineering Patterns

This notebook demonstrates key prompt engineering techniques for working with Large Language Models. No API key is needed -- we use templates and examples to illustrate the patterns.

## Topics Covered
1. Zero-shot prompting
2. Few-shot prompting
3. Chain-of-thought (CoT) prompting
4. Role prompting
5. System prompts
6. Structured output formatting
7. Reusable prompt template library
8. Comparison table of techniques

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import json
import textwrap

---
## 1. Zero-Shot Prompting

Ask the model to perform a task **without any examples**. Works well for straightforward tasks where the instruction is clear.

**When to use:** Simple classification, summarization, translation, or when examples are unavailable.

In [ ]:
# Zero-shot examples
zero_shot_prompts = {
    "Classification": {
        "prompt": """Classify the following text as either 'positive', 'negative', or 'neutral'.

Text: "The new restaurant downtown has amazing pasta but the service was quite slow."

Classification:""",
        "expected": "neutral (or mixed)"
    },
    "Summarization": {
        "prompt": """Summarize the following paragraph in one sentence.

Paragraph: Machine learning is a subset of artificial intelligence that focuses on building 
systems that can learn from data. Instead of being explicitly programmed with rules, these 
systems identify patterns in data and use them to make predictions or decisions. Common 
applications include image recognition, natural language processing, and recommendation systems.

Summary:""",
        "expected": "Machine learning is an AI approach where systems learn patterns from data to make predictions, with applications in image recognition, NLP, and recommendations."
    },
    "Extraction": {
        "prompt": """Extract all the programming languages mentioned in the following text.

Text: "Our team primarily uses Python for data science and machine learning tasks. 
The backend API is written in Go for performance, while the frontend uses TypeScript 
with React. We also maintain some legacy Java services."

Programming languages:""",
        "expected": "Python, Go, TypeScript, Java"
    }
}

for name, example in zero_shot_prompts.items():
    print(f"=== {name} ===")
    print(f"PROMPT:\n{example['prompt']}")
    print(f"\nEXPECTED RESPONSE: {example['expected']}")
    print("\n" + "-" * 70 + "\n")

---
## 2. Few-Shot Prompting

Provide **examples** ("shots") of the desired input-output format before the actual query. The model learns the pattern from these examples.

**When to use:** When the task requires a specific format, or zero-shot gives inconsistent results.

In [ ]:
few_shot_sentiment = """Classify the sentiment of each review.

Review: "This product exceeded all my expectations! Highly recommend."
Sentiment: positive

Review: "Terrible quality. Broke after one day of use."
Sentiment: negative

Review: "It works as described. Nothing special but gets the job done."
Sentiment: neutral

Review: "The battery life is incredible but the screen quality could be better."
Sentiment:"""

print("=== Few-Shot Sentiment Classification ===")
print(few_shot_sentiment)
print("\nExpected: neutral (or mixed)")
print("\nNote: The 3 examples above 'teach' the model the format and categories.")

In [ ]:
few_shot_entity = """Extract entities from the text in JSON format.

Text: "Apple CEO Tim Cook announced the new iPhone 15 in Cupertino on September 12."
Entities: {"company": "Apple", "person": "Tim Cook", "product": "iPhone 15", "location": "Cupertino", "date": "September 12"}

Text: "Elon Musk's Tesla delivered 435,000 vehicles in Q4 2023 from its Shanghai factory."
Entities: {"person": "Elon Musk", "company": "Tesla", "quantity": "435,000 vehicles", "date": "Q4 2023", "location": "Shanghai"}

Text: "Google released Gemini, their new AI model, at their Mountain View headquarters last Tuesday."
Entities:"""

print("=== Few-Shot Entity Extraction ===")
print(few_shot_entity)
print('\nExpected: {"company": "Google", "product": "Gemini", "location": "Mountain View", "date": "last Tuesday"}')

---
## 3. Chain-of-Thought (CoT) Prompting

Encourage the model to **show its reasoning step by step** before giving the final answer. This significantly improves performance on math, logic, and multi-step reasoning tasks.

**When to use:** Math problems, logic puzzles, multi-step reasoning, complex decisions.

In [ ]:
# Without CoT (often fails on complex math)
no_cot = """Q: A store sells apples for $2 each. If you buy 5 or more, you get a 20% discount. 
How much do 7 apples cost?

A:"""

# With CoT (much more reliable)
with_cot = """Q: A store sells apples for $2 each. If you buy 5 or more, you get a 20% discount. 
How much do 7 apples cost?

A: Let me think step by step.
1. Price per apple: $2
2. Number of apples: 7 (which is >= 5, so discount applies)
3. Full price: 7 x $2 = $14
4. Discount: 20% of $14 = $2.80
5. Final price: $14 - $2.80 = $11.20

The 7 apples cost $11.20."""

print("=== WITHOUT Chain-of-Thought ===")
print(no_cot)
print("(Model might jump to wrong answer)\n")
print("=" * 50)
print("\n=== WITH Chain-of-Thought ===")
print(with_cot)

In [ ]:
# CoT trigger phrases you can use
cot_triggers = [
    "Let's think step by step.",
    "Let's work through this carefully.",
    "Let me break this down:",
    "First, let's identify what we know...",
    "Let's reason about this:",
    "Think about this step by step before answering."
]

print("Common CoT trigger phrases:")
for i, trigger in enumerate(cot_triggers, 1):
    print(f"  {i}. \"{trigger}\"")

---
## 4. Role Prompting

Assign the model a specific **role or persona** to influence its response style, expertise level, and perspective.

**When to use:** When you need domain-specific language, a particular tone, or expert-level analysis.

In [ ]:
role_prompts = {
    "Senior Code Reviewer": {
        "prompt": """You are a senior software engineer conducting a code review. 
Analyze the following Python function for bugs, performance issues, and style.
Be specific and provide corrected code.

```python
def find_duplicates(lst):
    duplicates = []
    for i in range(len(lst)):
        for j in range(len(lst)):
            if i != j and lst[i] == lst[j]:
                if lst[i] not in duplicates:
                    duplicates.append(lst[i])
    return duplicates
```""",
        "note": "The role primes the model to think about performance (O(n^2)), suggest set-based approach, and follow code review conventions."
    },
    "Data Science Educator": {
        "prompt": """You are a patient data science instructor teaching beginners.
Explain what overfitting is using a simple real-world analogy.
Keep your explanation under 100 words.""",
        "note": "The 'patient instructor' role encourages simple language and analogies rather than technical jargon."
    },
    "Security Auditor": {
        "prompt": """You are a cybersecurity expert performing a security audit.
Review this API endpoint for security vulnerabilities:

```python
@app.route('/user/<user_id>')
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    result = db.execute(query)
    return jsonify(result)
```

List all vulnerabilities found and provide fixes.""",
        "note": "The security auditor role focuses the model on SQL injection, missing auth, data exposure, etc."
    }
}

for role, info in role_prompts.items():
    print(f"=== Role: {role} ===")
    print(f"PROMPT:\n{info['prompt']}")
    print(f"\nWHY THIS WORKS: {info['note']}")
    print("\n" + "-" * 70 + "\n")

---
## 5. System Prompts

System prompts set the **overall behavior, constraints, and personality** of the model for the entire conversation. They are processed differently from user messages by most API providers.

**When to use:** Always, when building applications. They define the assistant's identity and boundaries.

In [ ]:
system_prompt_examples = {
    "Customer Support Bot": {
        "system": """You are a helpful customer support assistant for TechCorp, 
a software company. Follow these rules:
1. Always be polite and empathetic
2. If you don't know the answer, say so and offer to escalate
3. Never share internal company information
4. Always ask if there's anything else you can help with
5. Keep responses concise (under 150 words)""",
        "user": "My account has been locked for 3 days and nobody is helping me!"
    },
    "SQL Assistant": {
        "system": """You are a SQL expert assistant. When given a question:
1. Write the SQL query
2. Explain what each part does
3. Mention any performance considerations
4. Always use parameterized queries (never string interpolation)
5. Default to PostgreSQL syntax unless told otherwise""",
        "user": "Find all customers who made more than 5 orders last month"
    },
    "JSON-Only Responder": {
        "system": """You are an API that ONLY responds in valid JSON. 
Never include any text outside the JSON object.
Always include a 'status' field ('success' or 'error') and a 'data' field.""",
        "user": "What is the capital of France?"
    }
}

for name, example in system_prompt_examples.items():
    print(f"=== {name} ===")
    print(f"SYSTEM: {example['system']}\n")
    print(f"USER: {example['user']}")
    print("\n" + "-" * 70 + "\n")

---
## 6. Structured Output Formatting

Techniques to get LLMs to produce consistently structured outputs (JSON, tables, lists, etc.).

In [ ]:
structured_output_techniques = {
    "JSON Schema Specification": {
        "prompt": """Analyze the following job posting and extract information in this exact JSON format:
{
    "title": "<job title>",
    "company": "<company name>",
    "location": "<location or 'remote'>",
    "salary_range": {"min": <number>, "max": <number>, "currency": "<USD/EUR/etc>"},
    "required_skills": ["<skill1>", "<skill2>"],
    "experience_years": <number>
}

Job posting: "We're hiring a Senior ML Engineer at DataCorp (San Francisco, hybrid). 
Salary: $180K-$250K. Requirements: 5+ years experience, Python, PyTorch, MLOps, Kubernetes."

JSON:""",
        "note": "Providing the exact schema with field types guides the model precisely."
    },
    "Markdown Table": {
        "prompt": """Compare Python, JavaScript, and Rust. 
Format your response as a markdown table with these columns:
| Language | Type System | Primary Use | Speed | Learning Curve |

Fill in each cell with 1-5 words maximum.""",
        "note": "Specifying column headers and constraints ensures consistent output."
    },
    "XML Tags": {
        "prompt": """Analyze the following text and provide your analysis using XML tags:

<text>The quarterly revenue increased by 15% but operating costs rose by 22%.</text>

Respond using these tags:
<summary>one sentence summary</summary>
<sentiment>positive/negative/neutral</sentiment>
<key_metrics>
  <metric name="...">value</metric>
</key_metrics>
<risk_level>low/medium/high</risk_level>""",
        "note": "XML tags are easy to parse programmatically and reduce ambiguity."
    }
}

for technique, info in structured_output_techniques.items():
    print(f"=== {technique} ===")
    print(f"PROMPT:\n{info['prompt']}")
    print(f"\nTIP: {info['note']}")
    print("\n" + "-" * 70 + "\n")

---
## 7. Reusable Prompt Template Library

A Python module for building prompts programmatically.

In [ ]:
@dataclass
class PromptTemplate:
    """Reusable prompt template with variable substitution."""
    name: str
    template: str
    system_prompt: Optional[str] = None
    examples: List[Dict[str, str]] = field(default_factory=list)
    
    def format(self, **kwargs) -> str:
        """Fill in template variables."""
        prompt = self.template.format(**kwargs)
        
        # Prepend examples if few-shot
        if self.examples:
            example_text = "\n\n".join(
                f"Input: {ex['input']}\nOutput: {ex['output']}" 
                for ex in self.examples
            )
            prompt = f"Examples:\n{example_text}\n\nNow do this:\n{prompt}"
        
        return prompt
    
    def build_messages(self, **kwargs) -> List[Dict[str, str]]:
        """Build a messages array suitable for chat API calls."""
        messages = []
        if self.system_prompt:
            messages.append({"role": "system", "content": self.system_prompt})
        messages.append({"role": "user", "content": self.format(**kwargs)})
        return messages


# Define reusable templates
templates = {}

# Sentiment analysis template
templates['sentiment'] = PromptTemplate(
    name="Sentiment Analysis",
    template="Classify the sentiment of this text as positive, negative, or neutral.\n\nText: \"{text}\"\n\nSentiment:",
    examples=[
        {"input": "I love this product!", "output": "positive"},
        {"input": "Worst purchase ever.", "output": "negative"},
        {"input": "It arrived on time.", "output": "neutral"},
    ]
)

# Summarization template
templates['summarize'] = PromptTemplate(
    name="Summarization",
    system_prompt="You are a concise summarizer. Always respond in {max_words} words or fewer.",
    template="Summarize the following text:\n\n{text}\n\nSummary:"
)

# Code review template
templates['code_review'] = PromptTemplate(
    name="Code Review",
    system_prompt="You are a senior software engineer. Review code for bugs, performance, style, and security.",
    template="""Review this {language} code:

```{language}
{code}
```

Provide:
1. Bugs found
2. Performance issues
3. Style improvements
4. Security concerns
5. Corrected code"""
)

# Chain-of-thought math template
templates['math_cot'] = PromptTemplate(
    name="Math (Chain-of-Thought)",
    template="""Solve this problem step by step. Show all work.

Problem: {problem}

Solution:
Let's think step by step."""
)

print("Template library created with templates:")
for name, tmpl in templates.items():
    print(f"  - {name}: {tmpl.name}")

In [ ]:
# Demonstrate using the templates
print("=== Using Sentiment Template (Few-Shot) ===")
prompt = templates['sentiment'].format(text="The movie was okay, nothing special.")
print(prompt)

print("\n" + "=" * 60 + "\n")

print("=== Using Code Review Template (with messages) ===")
messages = templates['code_review'].build_messages(
    language="python",
    code="""def login(username, password):\n    query = f"SELECT * FROM users WHERE name='{username}' AND pass='{password}'"\n    return db.execute(query)"""
)
for msg in messages:
    print(f"[{msg['role'].upper()}]")
    print(msg['content'])
    print()

print("=" * 60 + "\n")

print("=== Using Math CoT Template ===")
prompt = templates['math_cot'].format(
    problem="A train leaves City A at 9 AM traveling at 60 mph. Another train leaves City B "
            "(300 miles away) at 10 AM traveling toward City A at 90 mph. At what time do they meet?"
)
print(prompt)

---
## 8. Comparison Table of Techniques

| Technique | Description | Best For | Pros | Cons |
|-----------|------------|----------|------|------|
| **Zero-Shot** | No examples, just instructions | Simple, well-defined tasks | Fast, no example needed | May be inconsistent on complex tasks |
| **Few-Shot** | Provide 2-5 examples | Format-sensitive tasks, classification | Teaches format by example | Uses more tokens, examples may bias |
| **Chain-of-Thought** | Ask model to reason step-by-step | Math, logic, multi-step reasoning | Much better accuracy on reasoning | Slower, more tokens used |
| **Role Prompting** | Assign a persona/role | Domain-specific tasks, tone control | Focuses expertise, controls style | Role may introduce biases |
| **System Prompts** | Set global behavior rules | Application development | Persistent constraints, clean separation | Not all models support them equally |
| **Structured Output** | Specify exact output format | API integration, data extraction | Machine-parseable, consistent | May need validation/retry logic |

In [ ]:
# Summary as a structured data object for reference
techniques_summary = [
    {
        "technique": "Zero-Shot",
        "token_cost": "Low",
        "accuracy_boost": "Baseline",
        "complexity": "Simple",
        "best_for": ["Simple classification", "Translation", "Summarization"]
    },
    {
        "technique": "Few-Shot",
        "token_cost": "Medium",
        "accuracy_boost": "Moderate",
        "complexity": "Simple",
        "best_for": ["Format-specific tasks", "Consistent output style", "Classification with categories"]
    },
    {
        "technique": "Chain-of-Thought",
        "token_cost": "High",
        "accuracy_boost": "High",
        "complexity": "Moderate",
        "best_for": ["Math problems", "Logic puzzles", "Multi-step reasoning"]
    },
    {
        "technique": "Role Prompting",
        "token_cost": "Low-Medium",
        "accuracy_boost": "Moderate",
        "complexity": "Simple",
        "best_for": ["Domain expertise", "Tone control", "Specialized analysis"]
    },
    {
        "technique": "System Prompts",
        "token_cost": "Low",
        "accuracy_boost": "Moderate",
        "complexity": "Simple",
        "best_for": ["Application behavior", "Constraint enforcement", "Persona setting"]
    },
    {
        "technique": "Structured Output",
        "token_cost": "Medium",
        "accuracy_boost": "High (for parsing)",
        "complexity": "Moderate",
        "best_for": ["API responses", "Data extraction", "Machine-readable output"]
    }
]

print(json.dumps(techniques_summary, indent=2))

## Best Practices Summary

1. **Be specific** - Vague prompts get vague responses. Clearly state the task, format, and constraints.
2. **Use delimiters** - Separate instructions from content with triple quotes, XML tags, or markdown.
3. **Specify the output format** - Tell the model exactly what structure you want.
4. **Provide examples** when format matters - Few-shot is more reliable than lengthy descriptions.
5. **Use CoT for reasoning** - Always ask for step-by-step thinking on math/logic tasks.
6. **Iterate and refine** - Prompt engineering is empirical. Test, evaluate, and improve.
7. **Combine techniques** - Role + CoT + structured output often gives the best results.
8. **Keep system prompts focused** - Don't overload with too many rules.